# 02_model_10：公平比较版——相同特征、目标和验证流程下比较 Ridge 与 ExtraTrees

本版本专门回答一个问题：在相同的信息、相同的特征、相同的目标变换、相同的历史基线和相同的时间滚动验证流程下，ExtraTrees 是否仍然优于 Ridge。

公平增强版的两种模型共同使用：

- 相同的 236 个紧凑曲线、差分、历史天气和日历特征；
- 相同的历史基线 `B = w·D-1 + (1-w)·D-7`；
- 相同的比例目标 `Y/B - 1`；
- 相同的 4 折时间滚动验证；
- 相同的最终 2017-11-01 至 2017-12-30 测试集。

Ridge 只调 `alpha`，ExtraTrees 只调树模型超参数。这样比较到的是算法差异，而不是“一个模型做了特征工程、另一个模型仍使用旧输入”的差异。

## 1. 数据、预测任务与信息边界

目标仍然是预测目标日 `D` 的完整 144 点总负荷 `Z_total = Z1 + Z2 + Z3`。输入只使用预测时已经获得的信息：

- `D-1` 和 `D-7` 的历史负荷曲线；
- 目标日已知的星期、月份和周末特征；
- `D-1` 和 `D-7` 的历史天气观测摘要。

天气特征只来自过去日期，不使用目标日真实天气。两种公平增强版模型使用完全相同的天气特征。

In [34]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
FIG_DIR = 'figures'
SPLIT = pd.Timestamp('2017-11-01')
WEATHER_COLS = ['T', 'RH', 'WS', 'GDF', 'DF']
print('预测频率：10分钟；预测步长：1天；输出点数：144')

预测频率：10分钟；预测步长：1天；输出点数：144


In [ ]:
df = pd.read_csv('data/题4_城市电力负荷.csv', encoding='utf-8-sig')
df.columns = ['DateTime', 'T', 'RH', 'WS', 'GDF', 'DF', 'Z1', 'Z2', 'Z3']
df['DateTime'] = pd.to_datetime(df['DateTime'], format='%m/%d/%Y %H:%M')
df = df.set_index('DateTime').sort_index()
df['Z_total'] = df[['Z1', 'Z2', 'Z3']].sum(axis=1)

assert not df.index.duplicated().any()
step_minutes = df.index.to_series().diff().dropna().dt.total_seconds().div(60)
assert step_minutes.eq(10).all()
assert df.isna().sum().sum() == 0

day_profiles = {}
weather_profiles = {}
for day, group in df.groupby(df.index.normalize()):
    if len(group) == 144:
        day = pd.Timestamp(day)
        day_profiles[day] = group['Z_total'].to_numpy(dtype=float)
        weather_profiles[day] = {
            col: group[col].to_numpy(dtype=float) for col in WEATHER_COLS
        }

days = pd.DatetimeIndex(sorted(day_profiles))
print('完整天数:', len(days))
print('日期范围:', days.min().date(), '至', days.max().date())
print('采样间隔:', step_minutes.mode().iloc[0], '分钟')

## 2. 面向树模型的特征工程

原版本直接把 `D-1` 和 `D-7` 的 288 个负荷点输入树模型。这里保留历史信息，但同时构造更紧凑、更有业务含义的变量：

- 每条历史曲线按小时聚合为 24 个点；
- 构造 `D-1 − D-7` 的小时差分；
- 对 `D-1`、`D-7` 和差分曲线计算均值、标准差、极值、分位数和小时摘要；
- 加入过去两天的天气均值、标准差、极值及 D-1 与 D-7 的天气差异；
- 使用比例目标 `Y / B − 1`，其中 `B = w·D-1 + (1-w)·D-7`，让树模型学习相对历史基线的变化。

这里的 `w` 不由测试集决定，而是在训练期滚动验证中选择。

In [ ]:
def calendar_features(day):
    day = pd.Timestamp(day)
    return np.array([
        np.sin(2 * np.pi * day.dayofweek / 7),
        np.cos(2 * np.pi * day.dayofweek / 7),
        np.sin(2 * np.pi * (day.month - 1) / 12),
        np.cos(2 * np.pi * (day.month - 1) / 12),
        int(day.dayofweek >= 5),
    ], dtype=float)


def hourly_curve(curve):
    return np.asarray(curve, dtype=float).reshape(24, 6).mean(axis=1)


def curve_summary(curve):
    curve = np.asarray(curve, dtype=float)
    return np.r_[
        curve.mean(), curve.std(), curve.min(), curve.max(),
        np.quantile(curve, [0.10, 0.25, 0.50, 0.75, 0.90]),
        hourly_curve(curve),
    ]


def weather_summary(weather):
    arr = np.column_stack([weather[col] for col in WEATHER_COLS])
    return np.concatenate([
        np.array([arr[:, i].mean(), arr[:, i].std(), arr[:, i].min(), arr[:, i].max()])
        for i in range(arr.shape[1])
    ])

summary_labels = ['mean', 'std', 'min', 'max', 'q10', 'q25', 'q50', 'q75', 'q90'] + [f'hour_{i:02d}' for i in range(24)]
feature_names = (
    [f'D-1_hour_{i:02d}' for i in range(24)]
    + [f'D-7_hour_{i:02d}' for i in range(24)]
    + [f'diff_hour_{i:02d}' for i in range(24)]
    + [f'D-1_{x}' for x in summary_labels]
    + [f'D-7_{x}' for x in summary_labels]
    + [f'diff_{x}' for x in summary_labels]
    + [f'D-1_weather_{c}_{s}' for c in WEATHER_COLS for s in ['mean', 'std', 'min', 'max']]
    + [f'D-7_weather_{c}_{s}' for c in WEATHER_COLS for s in ['mean', 'std', 'min', 'max']]
    + [f'weather_diff_{c}_{s}' for c in WEATHER_COLS for s in ['mean', 'std', 'min', 'max']]
    + ['dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'is_weekend']
)

FEATURE_GROUPS = {
    'D-1小时曲线': (0, 24),
    'D-7小时曲线': (24, 48),
    '负荷差分小时曲线': (48, 72),
    'D-1曲线摘要': (72, 105),
    'D-7曲线摘要': (105, 138),
    '差分曲线摘要': (138, 171),
    'D-1天气摘要': (171, 191),
    'D-7天气摘要': (191, 211),
    '天气差分摘要': (211, 231),
    '日历特征': (231, 236),
}
assert len(feature_names) == 236


def optimized_features(target_day):
    d1 = day_profiles[target_day - pd.Timedelta(days=1)]
    d7 = day_profiles[target_day - pd.Timedelta(days=7)]
    diff = d1 - d7
    w1 = weather_summary(weather_profiles[target_day - pd.Timedelta(days=1)])
    w7 = weather_summary(weather_profiles[target_day - pd.Timedelta(days=7)])
    return np.r_[
        hourly_curve(d1), hourly_curve(d7), hourly_curve(diff),
        curve_summary(d1), curve_summary(d7), curve_summary(diff),
        w1, w7, w1 - w7,
        calendar_features(target_day),
    ]


def original_features(target_day):
    return np.r_[
        day_profiles[target_day - pd.Timedelta(days=1)],
        day_profiles[target_day - pd.Timedelta(days=7)],
        calendar_features(target_day),
    ]

X_original, X_optimized, Y = [], [], []
lag1_curves, lag7_curves, target_dates = [], [], []
for target_day in days:
    if target_day - pd.Timedelta(days=1) not in day_profiles:
        continue
    if target_day - pd.Timedelta(days=7) not in day_profiles:
        continue
    X_original.append(original_features(target_day))
    X_optimized.append(optimized_features(target_day))
    Y.append(day_profiles[target_day])
    lag1_curves.append(day_profiles[target_day - pd.Timedelta(days=1)])
    lag7_curves.append(day_profiles[target_day - pd.Timedelta(days=7)])
    target_dates.append(target_day)

X_original = np.asarray(X_original)
X_optimized = np.asarray(X_optimized)
Y = np.asarray(Y)
lag1_curves = np.asarray(lag1_curves)
lag7_curves = np.asarray(lag7_curves)
target_dates = pd.DatetimeIndex(target_dates)

train_mask = target_dates < SPLIT
test_mask = ~train_mask
test_dates = target_dates[test_mask]
print('原始特征:', X_original.shape, '| 优化特征:', X_optimized.shape)
print('X_train / X_test:', train_mask.sum(), '/', test_mask.sum())

## 3. 公平训练方案

先在训练期内部选择历史基线权重 `w`，再固定这个基线给两个模型使用。随后两个模型都预测同一个比例目标：

```text
B_D = w·负荷(D-1) + (1-w)·负荷(D-7)
Z_D = 真实负荷(D) / B_D − 1
预测负荷(D) = B_D × (1 + 模型预测的 Z_D)
```

两种公平增强版模型共享：

- `X_optimized`：同一个 236 维特征矩阵；
- `ratio_target`：同一个比例目标；
- `inner_cv`：同一组训练期时间滚动折；
- `Y_test`：同一个锁定测试集。

只允许模型自身的超参数不同：Ridge 调 `alpha`，ExtraTrees 调树深、叶节点样本数和特征采样比例。

表中的 `fold_RMSE` 是每个候选方案在 4 个时间验证折上的逐折 RMSE 明细；`CV_RMSE` 是这些折的平均值，`CV_RMSE_std` 是折间标准差。为了便于阅读，下面的展示表只显示平均值和标准差，逐折明细仍保留在变量中供检查。

In [ ]:
def smape(y_true, pred):
    return np.mean(2 * np.abs(pred - y_true) /
                   (np.abs(y_true) + np.abs(pred) + 1e-9)) * 100


def curve_rmse(y_true, pred):
    return float(np.sqrt(mean_squared_error(y_true, pred)))

# 原版本模型：仅作为历史参照，不用于“公平增强版”结论。
ridge_original = make_pipeline(StandardScaler(), Ridge(alpha=10.0))
ridge_original.fit(X_original[train_mask], Y[train_mask])
ridge_original_pred = ridge_original.predict(X_original[test_mask])

original_et = ExtraTreesRegressor(
    n_estimators=200, max_depth=12, min_samples_leaf=2,
    max_features=0.7, n_jobs=-1, random_state=0,
)
original_et.fit(X_original[train_mask], Y[train_mask])
original_et_pred = original_et.predict(X_original[test_mask])

train_indices = np.where(train_mask)[0]
inner_cv = TimeSeriesSplit(n_splits=4, test_size=30)

# 先用训练期验证选择一个共同的历史基线权重；该权重不由测试集选择。
baseline_rows = []
for weight in [0.0, 0.25, 0.50, 0.75, 1.0]:
    base = weight * lag1_curves + (1 - weight) * lag7_curves
    fold_scores = []
    for _, inner_valid in inner_cv.split(train_indices):
        valid_idx = train_indices[inner_valid]
        fold_scores.append(curve_rmse(Y[valid_idx], base[valid_idx]))
    baseline_rows.append({
        '前一天权重': weight,
        '前一周权重': 1 - weight,
        'CV_RMSE': np.mean(fold_scores),
        'CV_RMSE_std': np.std(fold_scores),
        'fold_RMSE': fold_scores,
    })
baseline_cv = pd.DataFrame(baseline_rows).sort_values('CV_RMSE').reset_index(drop=True)
selected_weight = float(baseline_cv.iloc[0]['前一天权重'])
print('共同历史基线的训练期选择结果：')
display(baseline_cv.drop(columns=['fold_RMSE']).style.format({'CV_RMSE': '{:,.2f}', 'CV_RMSE_std': '{:,.2f}'}))

common_base = selected_weight * lag1_curves + (1 - selected_weight) * lag7_curves
ratio_target = Y / (common_base + 1e-9) - 1

# 两个模型使用同一组折、同一个 ratio_target 和同一个 X_optimized。
ridge_alphas = [0.1, 1.0, 10.0, 30.0, 100.0, 300.0]
et_configs = [
    {'max_depth': 12, 'min_samples_leaf': 2, 'max_features': 0.7},
    {'max_depth': 12, 'min_samples_leaf': 5, 'max_features': 0.7},
    {'max_depth': 20, 'min_samples_leaf': 2, 'max_features': 0.7},
    {'max_depth': 20, 'min_samples_leaf': 5, 'max_features': 0.7},
    {'max_depth': 12, 'min_samples_leaf': 2, 'max_features': 1.0},
    {'max_depth': 12, 'min_samples_leaf': 5, 'max_features': 1.0},
    {'max_depth': 20, 'min_samples_leaf': 2, 'max_features': 1.0},
    {'max_depth': 20, 'min_samples_leaf': 5, 'max_features': 1.0},
]

ridge_cv_rows = []
for alpha in ridge_alphas:
    fold_scores = []
    for inner_train, inner_valid in inner_cv.split(train_indices):
        fit_idx = train_indices[inner_train]
        valid_idx = train_indices[inner_valid]
        model = make_pipeline(StandardScaler(), Ridge(alpha=alpha))
        model.fit(X_optimized[fit_idx], ratio_target[fit_idx])
        pred_ratio = model.predict(X_optimized[valid_idx])
        pred = common_base[valid_idx] * (1 + pred_ratio)
        fold_scores.append(curve_rmse(Y[valid_idx], pred))
    ridge_cv_rows.append({
        'alpha': alpha,
        'CV_RMSE': np.mean(fold_scores),
        'CV_RMSE_std': np.std(fold_scores),
        'fold_RMSE': fold_scores,
    })
ridge_cv_results = pd.DataFrame(ridge_cv_rows).sort_values('CV_RMSE').reset_index(drop=True)
best_ridge_alpha = float(ridge_cv_results.iloc[0]['alpha'])

et_cv_rows = []
for config in et_configs:
    fold_scores = []
    for inner_train, inner_valid in inner_cv.split(train_indices):
        fit_idx = train_indices[inner_train]
        valid_idx = train_indices[inner_valid]
        model = ExtraTreesRegressor(
            n_estimators=100, n_jobs=-1, random_state=0, **config
        )
        model.fit(X_optimized[fit_idx], ratio_target[fit_idx])
        pred_ratio = model.predict(X_optimized[valid_idx])
        pred = common_base[valid_idx] * (1 + pred_ratio)
        fold_scores.append(curve_rmse(Y[valid_idx], pred))
    et_cv_rows.append({
        **config,
        'CV_RMSE': np.mean(fold_scores),
        'CV_RMSE_std': np.std(fold_scores),
        'fold_RMSE': fold_scores,
    })
et_cv_results = pd.DataFrame(et_cv_rows).sort_values('CV_RMSE').reset_index(drop=True)
best_et_row = et_cv_results.iloc[0]
best_et_config = {
    'max_depth': int(best_et_row['max_depth']),
    'min_samples_leaf': int(best_et_row['min_samples_leaf']),
    'max_features': float(best_et_row['max_features']),
}

print('公平 Ridge 的训练期调参结果：')
display(ridge_cv_results.drop(columns=['fold_RMSE']).style.format({'CV_RMSE': '{:,.2f}', 'CV_RMSE_std': '{:,.2f}'}))
print('公平 ExtraTrees 的训练期调参结果：')
display(et_cv_results.drop(columns=['fold_RMSE']).style.format({'CV_RMSE': '{:,.2f}', 'CV_RMSE_std': '{:,.2f}'}))

# 两个公平增强版在完全相同的输入/目标/基线下重新训练。
fair_ridge = make_pipeline(StandardScaler(), Ridge(alpha=best_ridge_alpha))
fair_ridge.fit(X_optimized[train_mask], ratio_target[train_mask])
fair_ridge_pred = common_base[test_mask] * (1 + fair_ridge.predict(X_optimized[test_mask]))

fair_et = ExtraTreesRegressor(
    n_estimators=500, n_jobs=-1, random_state=0, **best_et_config
)
fair_et.fit(X_optimized[train_mask], ratio_target[train_mask])
fair_et_pred = common_base[test_mask] * (1 + fair_et.predict(X_optimized[test_mask]))
fair_et_pred = np.maximum(fair_et_pred, 0)

predictions = {
    'Ridge原版': ridge_original_pred,
    'ExtraTrees原版': original_et_pred,
    '共同历史基线': common_base[test_mask],
    '公平Ridge': fair_ridge_pred,
    '公平ExtraTrees': fair_et_pred,
}
print('共同历史基线前一天权重:', selected_weight)
print('公平 Ridge alpha:', best_ridge_alpha)
print('公平 ExtraTrees 参数:', best_et_config)

## 4. 公平测试集比较

最终结论只比较“公平 Ridge”和“公平 ExtraTrees”。原版模型和共同历史基线用于说明优化前后的变化，但不能用于证明某个算法在相同条件下更好。

如果公平 ExtraTrees 的测试误差仍低于公平 Ridge，才可以说：在当前特征、目标变换、历史天气信息和测试时间窗下，树模型的非线性修正优于线性修正。

In [ ]:
def curve_metric_row(name, y_true, pred):
    mse = mean_squared_error(y_true, pred)
    return {
        '模型': name,
        '曲线MSE(kW²)': mse,
        '曲线RMSE(kW)': np.sqrt(mse),
        '曲线MAE(kW)': mean_absolute_error(y_true, pred),
        '曲线sMAPE%': smape(y_true, pred),
        '曲线归一化RMSE%': np.sqrt(mse) / np.mean(np.abs(y_true)) * 100,
        '曲线逐时间点平均R2': r2_score(y_true, pred),
        '曲线展平R2': r2_score(y_true.ravel(), pred.ravel()),
    }

curve_metrics = pd.DataFrame([
    curve_metric_row(name, Y[test_mask], pred)
    for name, pred in predictions.items()
])
display(curve_metrics.style.format({
    '曲线MSE(kW²)': '{:,.0f}', '曲线RMSE(kW)': '{:,.2f}', '曲线MAE(kW)': '{:,.2f}',
    '曲线sMAPE%': '{:.2f}%', '曲线归一化RMSE%': '{:.2f}%',
    '曲线逐时间点平均R2': '{:.4f}', '曲线展平R2': '{:.4f}',
}))

summary_rows = []
for name, pred in predictions.items():
    for label, fn in [
        ('日用电量(kWh)', lambda x: x.sum(axis=1) / 6),
        ('日峰值(kW)', lambda x: x.max(axis=1)),
        ('日谷值(kW)', lambda x: x.min(axis=1)),
    ]:
        yt, yp = fn(Y[test_mask]), fn(pred)
        summary_rows.append({
            '模型': name,
            '目标': label,
            'MSE': mean_squared_error(yt, yp),
            'RMSE': np.sqrt(mean_squared_error(yt, yp)),
            'MAE': mean_absolute_error(yt, yp),
            'sMAPE%': smape(yt, yp),
            'R2': r2_score(yt, yp),
        })
summary_metrics = pd.DataFrame(summary_rows)
display(summary_metrics.style.format({
    'MSE': '{:,.0f}', 'RMSE': '{:,.2f}', 'MAE': '{:,.2f}', 'sMAPE%': '{:.2f}%', 'R2': '{:.4f}'
}))

## 5. 公平 ExtraTrees 的特征组重要性

特征重要性只用于解释最终公平 ExtraTrees 使用了哪些特征组，不用于测试集筛选。由于特征之间存在相关性，importance 排名不能单独证明因果关系；若继续做特征选择，应在训练期验证折上使用分组 permutation importance。

In [ ]:
importance_rows = []
for group, (start, end) in FEATURE_GROUPS.items():
    importance_rows.append({
        '特征组': group,
        '重要性': fair_et.feature_importances_[start:end].sum(),
    })
importance_df = pd.DataFrame(importance_rows).sort_values('重要性', ascending=False)
display(importance_df.style.format({'重要性': '{:.4f}'}))

## 6. 可视化

In [ ]:
example_idx = min(10, len(test_dates) - 1)
slot_axis = np.arange(144) / 6
fig, ax = plt.subplots(figsize=(13, 4.8))
ax.plot(slot_axis, Y[test_mask][example_idx], color='black', lw=1.6, label='真实负荷')
for name, color in [
    ('Ridge原版', '#1f77b4'),
    ('ExtraTrees原版', '#ff7f0e'),
    ('公平Ridge', '#2ca02c'),
    ('公平ExtraTrees', '#d62728'),
    ('共同历史基线', '#9467bd'),
]:
    ax.plot(slot_axis, predictions[name][example_idx], lw=1.1, label=name, alpha=0.9)
ax.set_title(f'图1  {test_dates[example_idx].date()}：公平模型完整日曲线')
ax.set_xlabel('时刻'); ax.set_ylabel('负荷 (kW)')
ax.set_xticks(range(0, 25, 2)); ax.grid(alpha=0.3); ax.legend(); fig.tight_layout()
fig.savefig(f'{FIG_DIR}/fig_B10_curve.png', dpi=160, bbox_inches='tight'); plt.show()

fig, ax = plt.subplots(figsize=(13, 4.8))
actual_energy = Y[test_mask].sum(axis=1) / 6
ax.plot(test_dates, actual_energy, color='black', lw=1.5, label='真实日用电量')
for name, color in [('Ridge原版', '#1f77b4'), ('ExtraTrees原版', '#ff7f0e'), ('公平Ridge', '#2ca02c'), ('公平ExtraTrees', '#d62728')]:
    ax.plot(test_dates, predictions[name].sum(axis=1) / 6, lw=1.1, label=name)
ax.set_title('图2  测试期日用电量预测')
ax.set_xlabel('目标日期'); ax.set_ylabel('日用电量 (kWh)')
ax.grid(alpha=0.3); ax.legend(); fig.tight_layout()
fig.savefig(f'{FIG_DIR}/fig_B10_energy.png', dpi=160, bbox_inches='tight'); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
curve_metrics.plot(x='模型', y='曲线RMSE(kW)', kind='bar', ax=axes[0], legend=False,
                   color=['#1f77b4', '#ff7f0e', '#9467bd', '#2ca02c', '#d62728'])
axes[0].set_title('曲线 RMSE 对比'); axes[0].set_ylabel('RMSE (kW)'); axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=25); axes[0].grid(alpha=0.25, axis='y')
importance_df.plot(x='特征组', y='重要性', kind='bar', ax=axes[1], legend=False, color='#8c564b')
axes[1].set_title('公平 ExtraTrees 的特征组重要性'); axes[1].set_ylabel('重要性'); axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=60); axes[1].grid(alpha=0.25, axis='y')
fig.suptitle('图3  公平模型比较与特征解释', y=1.02); fig.tight_layout()
fig.savefig(f'{FIG_DIR}/fig_B10_metrics_importance.png', dpi=160, bbox_inches='tight'); plt.show()

## 7. 公平结论与过拟合边界

本版本的核心比较是公平 Ridge 与公平 ExtraTrees。两者使用相同的 236 个输入特征、相同的历史天气信息、相同的比例目标、相同的历史基线和相同的时间滚动验证折。

- 如果公平 ExtraTrees 仍优于公平 Ridge，可以把结论写成“在当前任务和信息边界下，树模型对相对历史基线的修正更有效”；
- 如果公平 Ridge 更好，则说明前一版的提升主要来自特征工程和目标变换，而不是树模型本身；
- 原版 Ridge、原版 ExtraTrees 与公平模型的差异，只用于展示优化收益，不用于算法公平性结论。

调参只使用训练期滚动验证，最终测试集只评估一次。但由于本项目只有一年数据，且候选方案有限，跨年份泛化仍需更多年份和滚动外部测试确认。特征重要性用于解释，不能替代独立验证。

## 8. 测试集逐日误差与代表性日期

单独展示一个测试日容易造成选择偏差。下面增加两组图：

1. 用 60 个测试日的逐日 RMSE、RMSE 差值和经验分布展示整体胜负；
2. 按预先定义的规则展示三个日期：ExtraTrees 优势最大的日期、两者最接近的日期、Ridge 优势最大的日期。

其中 `ExtraTrees − Ridge < 0` 表示 ExtraTrees 当天更好。代表性日期来自测试集结果，只用于解释误差形态，不能替代总体指标，也不能只展示树模型获胜日期。

In [ ]:
y_test = Y[test_mask]
ridge_day_rmse = np.sqrt(np.mean((predictions['公平Ridge'] - y_test) ** 2, axis=1))
et_day_rmse = np.sqrt(np.mean((predictions['公平ExtraTrees'] - y_test) ** 2, axis=1))
daily_compare = pd.DataFrame({
    '日期': test_dates,
    '公平Ridge日RMSE': ridge_day_rmse,
    '公平ExtraTrees日RMSE': et_day_rmse,
})
daily_compare['ExtraTrees-Ridge日RMSE差'] = daily_compare['公平ExtraTrees日RMSE'] - daily_compare['公平Ridge日RMSE']
daily_compare['ExtraTrees是否更好'] = daily_compare['ExtraTrees-Ridge日RMSE差'] < 0
print('ExtraTrees逐日胜出:', int(daily_compare['ExtraTrees是否更好'].sum()), '/', len(daily_compare))
print('Ridge逐日胜出:', int((~daily_compare['ExtraTrees是否更好']).sum()), '/', len(daily_compare))
print('逐日RMSE中位数：Ridge={:.2f}, ExtraTrees={:.2f}'.format(np.median(ridge_day_rmse), np.median(et_day_rmse)))
display(daily_compare.head())

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes[0, 0].boxplot([ridge_day_rmse, et_day_rmse], tick_labels=['公平Ridge', '公平ExtraTrees'], showmeans=True)
axes[0, 0].set_title('60个测试日的逐日 RMSE 分布'); axes[0, 0].set_ylabel('日 RMSE (kW)'); axes[0, 0].grid(alpha=0.25, axis='y')
axes[0, 1].scatter(ridge_day_rmse, et_day_rmse, alpha=0.75, color='#d62728')
lim = max(ridge_day_rmse.max(), et_day_rmse.max()) * 1.05
axes[0, 1].plot([0, lim], [0, lim], 'k--', lw=1); axes[0, 1].set_xlim(0, lim); axes[0, 1].set_ylim(0, lim)
axes[0, 1].set_xlabel('公平 Ridge 日 RMSE (kW)'); axes[0, 1].set_ylabel('公平 ExtraTrees 日 RMSE (kW)')
axes[0, 1].set_title('逐日 RMSE：点在对角线下方表示树模型更好'); axes[0, 1].grid(alpha=0.25)
sorted_delta = daily_compare.sort_values('ExtraTrees-Ridge日RMSE差')
colors = ['#d62728' if v < 0 else '#2ca02c' for v in sorted_delta['ExtraTrees-Ridge日RMSE差']]
axes[1, 0].bar(np.arange(len(sorted_delta)), sorted_delta['ExtraTrees-Ridge日RMSE差'], color=colors)
axes[1, 0].axhline(0, color='black', lw=1); axes[1, 0].set_title('逐日 RMSE 差值：ExtraTrees - Ridge')
axes[1, 0].set_xlabel('测试日（按差值排序）'); axes[1, 0].set_ylabel('RMSE 差 (kW)'); axes[1, 0].grid(alpha=0.25, axis='y')
for vals, label, color in [(ridge_day_rmse, '公平Ridge', '#2ca02c'), (et_day_rmse, '公平ExtraTrees', '#d62728')]:
    x = np.sort(vals); y = np.arange(1, len(x) + 1) / len(x); axes[1, 1].step(x, y, where='post', label=label, color=color)
axes[1, 1].set_title('逐日 RMSE 经验累积分布'); axes[1, 1].set_xlabel('日 RMSE (kW)'); axes[1, 1].set_ylabel('累计比例'); axes[1, 1].grid(alpha=0.25); axes[1, 1].legend()
fig.suptitle('图4  公平模型的逐日误差分布', y=1.02); fig.tight_layout(); fig.savefig(f'{FIG_DIR}/fig_B10_error_distribution.png', dpi=160, bbox_inches='tight'); plt.show()

slot_axis = np.arange(144) / 6
delta = daily_compare['ExtraTrees-Ridge日RMSE差'].to_numpy()
representative = [(int(np.argmin(delta)), 'ExtraTrees优势最大'), (int(np.argmin(np.abs(delta))), '两模型最接近'), (int(np.argmax(delta)), 'Ridge优势最大')]
fig, axes = plt.subplots(3, 1, figsize=(13, 11), sharex=True)
for ax, (idx, label) in zip(axes, representative):
    ax.plot(slot_axis, y_test[idx], color='black', lw=1.6, label='真实负荷')
    ax.plot(slot_axis, predictions['公平Ridge'][idx], color='#2ca02c', lw=1.2, label='公平Ridge')
    ax.plot(slot_axis, predictions['公平ExtraTrees'][idx], color='#d62728', lw=1.2, label='公平ExtraTrees')
    ax.set_ylabel('负荷 (kW)'); ax.set_title('{}：{}；Ridge日RMSE={:.0f}，ExtraTrees日RMSE={:.0f}'.format(test_dates[idx].date(), label, ridge_day_rmse[idx], et_day_rmse[idx])); ax.grid(alpha=0.25)
axes[-1].set_xlabel('时刻'); axes[-1].set_xticks(range(0, 25, 2)); axes[0].legend(ncol=3)
fig.suptitle('图5  按固定规则选出的三类测试日示例', y=1.01); fig.tight_layout(); fig.savefig(f'{FIG_DIR}/fig_B10_representative_days.png', dpi=160, bbox_inches='tight'); plt.show()

## 9. 评分点对应与最终深度结论

### 9.1 数据分析的全面性与深度

分析对象是三区合计总负荷 `Z_total`，因为它更接近全市调度和电源侧的实际对象，也能减少单一区域的随机波动。数据保留 10 分钟粒度，每个完整日有 144 个点；原始数据共有 364 个完整日，目标日因需要 D-1 和 D-7 历史信息，从 2017-01-08 开始构造样本。训练集为 2017-01-08 至 2017-10-31，测试集为 2017-11-01 至 2017-12-30，共 297 个训练日和 60 个测试日。

输入同时包含近期惯性、周周期、目标日历和历史天气信息。前一天负荷反映最近状态，前一周负荷反映同星期结构；小时压缩、曲线摘要和差分特征帮助树模型学习负荷水平、形状和变化。需要说明的是，当前数据只有一年，测试期处于训练期之后，存在明显季节分布变化，因此泛化结论仍然有限。

### 9.2 方法选择、运用与评估质量

本项目同时比较原版 Ridge、原版 ExtraTrees，以及在公平条件下使用同一组 236 个特征、同一历史基线、同一比例目标和同一时间滚动验证的公平 Ridge 与公平 ExtraTrees。Ridge 通过标准化和 L2 正则化处理小样本、高维和强共线输入；ExtraTrees 用于表达非线性和特征交互。

最终评价使用 MSE、RMSE、MAE、sMAPE、归一化 RMSE 和两种明确标注的 R²口径。调参只在训练期滚动验证中进行，测试集只在参数锁定后使用一次。原始 ExtraTrees 表现较差，但经过基线比例目标、紧凑曲线特征、历史天气摘要和时间滚动调参后，公平 ExtraTrees 的性能明显提升。

### 9.3 可视化有效性

测试日曲线图用于观察峰谷、爬坡和下降过程；测试期日用电量图用于观察日级偏差；指标图用于横向比较；逐日误差分布图展示 60 天的总体胜负、误差中位数和大误差情况；代表性日期图同时包含 ExtraTrees 优势日、两模型接近日和 Ridge 优势日，避免只挑选有利于某个模型的日期。特征组重要性图用于解释模型使用的信息，但不作为单独的因果证据。

### 9.4 结论论证、独立思考与创新性

本项目的主要独立思考是：树模型不直接从零预测完整负荷，而是先利用历史负荷形成可解释的基线，再学习相对基线的比例修正。这种目标重构降低了树模型对季节水平外推的压力。把历史曲线压缩为小时形状、摘要和差分，并加入只在预测时可获得的 D-1/D-7 天气摘要，是针对本数据结构的特征设计。

最终结论应分两层表述：原版直接 ExtraTrees 不如原版 Ridge；在相同增强特征、相同目标、相同基线和相同滚动验证下，公平 ExtraTrees 在当前测试集上的总体 RMSE 低于公平 Ridge。这个结果支持“当前任务中的非线性修正有额外价值”，但不能推广为树模型在所有时期、所有数据集上都优于线性模型。